# 🧪 The Prompt Optimization Loop

You've collected user feedback, turned thumbs-down responses into signals — now it's time to close the loop:
use that feedback to **systematically improve your Canopy summarization prompt**.

```
Register Prompt  →  Run Queries  →  Attach Feedback
       ↑                                    ↓
  New Version  ←  Optimize  ←  Evaluate  ←  Build Dataset
```

**What you'll do in this notebook:**
1. **Register** the current Canopy summarization prompt in MLflow's Prompt Registry
2. **Run** sample summarization queries — with traces automatically linked to the prompt version
3. **Attach** feedback (pull real thumbs-down from Canopy, or use simulated data)
4. **Build** an evaluation dataset from those annotated traces
5. **Evaluate** how well the current prompt performs against your expectations
6. **Optimize** — let MLflow's GEPA automatically rewrite the prompt based on failures
7. **Compare** v1 vs optimized side-by-side and decide whether to promote

---
## 0. Setup

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import mlflow
from mlflow.entities import AssessmentSource, AssessmentSourceType
from mlflow.genai.scorers import scorer
from mlflow.genai.optimize import GepaPromptOptimizer
from openai import OpenAI
import pandas as pd

from support_functions import get_namespace

> ⚠️ **Note:** Update the variables below to match your cluster and model endpoint.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CONFIGURATION — update these to match your environment
# ─────────────────────────────────────────────────────────────────

# MLflow server
MLFLOW_TRACKING_URI   = "https://rh-ai.<CLUSTER-DOMAIN>/mlflow"
MLFLOW_TRACKING_TOKEN = "<OPENSHIFT-API-TOKEN>"

# MLflow names (feel free to keep these as-is)
EXPERIMENT_NAME = "prompt-optimization-loop"
PROMPT_NAME     = "summarization"
DATASET_NAME    = "summarization-eval"

# LLM — point at the same model Canopy uses
LLM_API_KEY  = "<MODEL-API-KEY>"
LLM_BASE_URL = "<MODEL-ENDPOINT>"  # e.g. http://llama-32-predictor:80/v1
LLM_MODEL    = "<MODEL-NAME>"      # e.g. llama32

In [ ]:
os.environ["MLFLOW_TRACKING_TOKEN"]       = MLFLOW_TRACKING_TOKEN
os.environ["MLFLOW_WORKSPACE"]            = get_namespace()
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"✓ Connected to MLflow — experiment: '{EXPERIMENT_NAME}' ({experiment_id})")
print(f"  → {MLFLOW_TRACKING_URI}")

---
## Step 1: Register the Summarization Prompt

We start by registering the current Canopy summarization prompt in the **MLflow Prompt Registry**.
Every call to `register_prompt` creates a new immutable version — you can always diff, reload, or roll back.

This baseline is intentionally minimal, which gives the optimizer room to improve it.

In [ ]:
# This matches what's currently deployed in Canopy
SYSTEM_PROMPT_V1 = """Summarize the following text in a clear and detailed manner:"""

prompt_v1 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=SYSTEM_PROMPT_V1,
    commit_message="v1: baseline summarization prompt",
    tags={"version_type": "baseline"},
)

PROMPT_V1_URI   = f"prompts:/{PROMPT_NAME}/{prompt_v1.version}"
PROMPT_PROD_URI = f"prompts:/{PROMPT_NAME}@prod"

mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="prod", version=prompt_v1.version)

print(f"✓ Registered '{PROMPT_NAME}' as version {prompt_v1.version}")
print(f"  → Aliased 'prod' → v{prompt_v1.version}")
print(f"  → URI: {PROMPT_V1_URI}")

---
## Step 2: Run Summarization Queries

We load the prompt via `mlflow.genai.load_prompt()` **inside** the `@mlflow.trace` decorator.
This is what links each trace to the specific prompt version — so later you can filter
traces by prompt in the MLflow UI and see exactly which version produced which output.

The five sample texts below cover different topics. Using varied inputs makes the evaluation more robust.

In [ ]:
SAMPLE_TEXTS = [
    """Collecting constant user feedback is one of the most reliable ways to make sure you're building 
the right thing — and building it well. It helps you validate assumptions early, catch usability issues 
before they turn into costly rework, and prioritize improvements based on real-world needs instead of 
internal guesses. A steady feedback loop also builds trust: when users see their input reflected in 
updates, they feel heard and become more willing to engage.""",

    """Machine learning models degrade over time as the real-world distribution of data shifts away from 
what the model was trained on. This phenomenon — called data drift or concept drift — is one of the 
primary reasons why production ML systems require continuous monitoring and periodic retraining. 
Without drift detection, a model that performs well on day one may silently produce poor results 
months later, with no obvious error signal.""",

    """Observability in distributed systems refers to the ability to understand the internal state of a 
system by examining its outputs. The three pillars of observability are metrics (quantitative 
measurements over time), logs (timestamped records of events), and traces (records of requests 
as they flow through distributed components). Together they give teams the context needed to debug 
issues, optimize performance, and maintain reliability in complex environments.""",

    """Retrieval-Augmented Generation (RAG) combines the power of large language models with external 
knowledge retrieval. Instead of relying solely on knowledge baked into model weights during training, 
RAG systems first search a knowledge base for relevant context, then provide that context to the model 
along with the user's query. This reduces hallucinations and makes it possible to build applications 
over private or frequently updated data without retraining.""",

    """GitOps is an operational framework that applies DevOps best practices — version control, 
collaboration, and CI/CD — to infrastructure automation. In a GitOps workflow, the desired state 
of the infrastructure is stored in a Git repository, and automated processes continuously reconcile 
the actual state with the desired state. This gives teams a single source of truth, a full audit 
trail of every change, and the ability to roll back by reverting a commit.""",
]

mlflow.openai.autolog()

llm_client = OpenAI(
    api_key=LLM_API_KEY,
    **(dict(base_url=LLM_BASE_URL) if LLM_BASE_URL else {}),
)

_last_trace_id: list = []


@mlflow.trace(name="summarize")
def summarize_traced(text: str, prompt_uri: str) -> str:
    """
    Summarize text using a prompt loaded from the registry.
    load_prompt() inside @mlflow.trace links this trace to the prompt version.
    """
    _last_trace_id.clear()
    _last_trace_id.append(mlflow.get_active_trace_id())

    prompt = mlflow.genai.load_prompt(prompt_uri)  # ← prompt linkage

    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": prompt.template},
            {"role": "user",   "content": text},
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content

In [ ]:
v1_responses = []  # stores {text, summary, trace_id}

for i, text in enumerate(SAMPLE_TEXTS, 1):
    summary  = summarize_traced(text, PROMPT_V1_URI)
    trace_id = _last_trace_id[0] if _last_trace_id else None
    v1_responses.append({"text": text, "summary": summary, "trace_id": trace_id})

    print(f"[{i}] {text[:70].strip()}...")
    print(f"  → {summary}")
    print(f"  trace_id: {trace_id}\n")

---
## Step 3: Attach Feedback to Traces

This is where the Canopy feedback loop and the prompt optimization loop connect.

You have two options:

- **Option A** — pull real thumbs-down traces directly from the Canopy `summarization` experiment.  
  Use this if you've been running Canopy with feedback enabled and have 👎 data to work with.
- **Option B** — attach simulated feedback to the traces we just ran in Step 2.  
  Use this if you're starting fresh or want a fully controlled experiment.

Either way the output is the same: traces with `user_satisfaction` feedback attached, ready for the eval dataset.

### Option A: Pull Real Negative Feedback from Canopy

If you've been using Canopy with `feedback: enabled: true`, your thumbs-down traces
are already sitting in MLflow. We can pull them directly:

In [ ]:
real_negative_traces = mlflow.search_traces(
    experiment_names=["summarization"],
    filter_string="assessments.user_satisfaction.value = 'false'",
    max_results=20,
    return_type="list",
)

if real_negative_traces:
    print(f"Found {len(real_negative_traces)} thumbs-down traces from Canopy.")
    print("You can merge these directly into the eval dataset in Step 4 — skip Option B.")
    for t in real_negative_traces[:3]:
        print(f"  trace_id={t.info.request_id}")
    if len(real_negative_traces) > 3:
        print(f"  ... and {len(real_negative_traces) - 3} more")
else:
    print("No thumbs-down traces found in 'summarization'. Proceed with Option B below.")

### Option B: Simulate Feedback

Attach simulated 👍 / 👎 to the traces we ran in Step 2.  
In production this would come from users clicking the feedback buttons in Canopy — here we replicate that signal.

In [ ]:
SIMULATED_FEEDBACK = [
    {"rating": True,  "comment": "Good summary, covers the main points"},
    {"rating": False, "comment": "Too vague — didn't mention the types of drift"},
    {"rating": True,  "comment": "Clear explanation of the three pillars"},
    {"rating": False, "comment": "Didn't spell out what RAG stands for"},
    {"rating": True,  "comment": "Captured the GitOps workflow well"},
]

print("Attaching feedback to traces...\n")

for resp, fb in zip(v1_responses, SIMULATED_FEEDBACK):
    trace_id = resp["trace_id"]
    if not trace_id:
        print(f"  ⚠ No trace ID for: {resp['text'][:50]}")
        continue

    mlflow.log_feedback(
        trace_id=trace_id,
        name="user_satisfaction",
        value=fb["rating"],
        rationale=fb["comment"],
        source=AssessmentSource(
            source_type=AssessmentSourceType.HUMAN,
            source_id="notebook_user",
        ),
    )

    icon = "👍" if fb["rating"] else "👎"
    print(f"  {icon}  {resp['text'][:65].strip()}...")
    print(f"       '{fb['comment']}'")

print("\n✓ Feedback attached!")

### Add Expectations

Expectations are the ground truth: what a correct summary *must* include.  
They're stored on the traces and become the scoring criteria in Steps 5 and 6.

In [ ]:
EXPECTATIONS = [
    {"must_contain": "feedback",      "max_chars": 250},
    {"must_contain": "drift",         "max_chars": 250},
    {"must_contain": "observability", "max_chars": 250},
    {"must_contain": "retrieval",     "max_chars": 250},
    {"must_contain": "gitops",        "max_chars": 250},
]

print("Attaching expectations to traces...\n")

for resp, exp in zip(v1_responses, EXPECTATIONS):
    trace_id = resp["trace_id"]
    if not trace_id:
        continue

    mlflow.log_expectation(
        trace_id=trace_id,
        name="expected_keyword",
        value=exp["must_contain"],
        source=AssessmentSource(source_type=AssessmentSourceType.HUMAN, source_id="annotator"),
    )
    mlflow.log_expectation(
        trace_id=trace_id,
        name="expected_max_chars",
        value=str(exp["max_chars"]),
        source=AssessmentSource(source_type=AssessmentSourceType.HUMAN, source_id="annotator"),
    )
    print(f"  [{exp['must_contain']:<14} ≤{exp['max_chars']}ch]  {resp['text'][:55].strip()}...")

print("\n✓ Expectations attached!")

---
## Step 4: Build the Evaluation Dataset

We collect the annotated traces into an MLflow dataset.
Because it's hosted in MLflow, every future evaluation run is linked to it —
you can always trace which data produced which result.

In [ ]:
our_trace_ids = {r["trace_id"] for r in v1_responses if r["trace_id"]}

all_traces = mlflow.search_traces(
    experiment_ids=[experiment_id],
    max_results=50,
    return_type="list",
)

our_traces = [t for t in all_traces if t.info.request_id in our_trace_ids]
print(f"Found {len(our_traces)} annotated traces")

eval_dataset = mlflow.genai.create_dataset(
    name=DATASET_NAME,
    experiment_id=experiment_id,
    tags={"source": "annotated-traces", "version": "1"},
)
eval_dataset.merge_records(our_traces)

print(f"✓ Dataset '{DATASET_NAME}' created with {len(our_traces)} records")

---
## Step 5: Evaluate Prompt v1

`mlflow.genai.evaluate()` runs every record in the dataset through the model and scores the output.

We use two simple scorers:
- **`contains_keyword`** — does the summary mention the required keyword from the expectations?
- **`is_concise`** — is the summary under the expected character limit?

In [ ]:
@scorer
def contains_keyword(inputs: dict, outputs: str, expectations: dict) -> bool:
    """Does the summary contain the required keyword?"""
    if not outputs or not expectations:
        return False
    return expectations.get("must_contain", "").lower() in outputs.lower()


@scorer
def is_concise(outputs: str, expectations: dict) -> bool:
    """Is the summary under the character limit?"""
    if not outputs:
        return False
    return len(outputs) <= expectations.get("max_chars", 300)


print("✓ Scorers defined: contains_keyword, is_concise")

In [ ]:
def predict_v1(text: str, **kwargs) -> str:
    return summarize_traced(text, PROMPT_V1_URI)


print("Running evaluation with prompt v1...")
eval_v1 = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_v1,
    scorers=[contains_keyword, is_concise],
)

print("\n── Evaluation Results: Prompt v1 ──")
for metric, value in sorted(eval_v1.metrics.items()):
    bar = "█" * int((value if isinstance(value, float) else 0) * 20)
    print(f"  {metric:<35} {f'{value:.0%}' if isinstance(value, float) else value}  {bar}")

---
## Step 6: Optimize with GEPA

**GEPA (Generate, Evaluate, Predict, Adapt)** is MLflow's automated prompt optimizer.

How it works:
1. Runs `predict_fn` on the training data to collect initial scores
2. Uses a *reflection model* to analyse which examples failed and why
3. Generates a rewritten prompt that addresses those weaknesses
4. Registers the improved prompt as a new version in the Prompt Registry

The reflection model analyses failures — it can be the same model you use for inference,
or a more capable one if you want deeper analysis.

> ⏱️ This takes a few minutes — the optimizer runs multiple evaluation passes internally.

In [ ]:
if LLM_BASE_URL:
    os.environ["OPENAI_BASE_URL"] = LLM_BASE_URL
os.environ["OPENAI_API_KEY"] = LLM_API_KEY


def predict_fn(text: str, **kwargs) -> str:
    """
    Predict function for the optimizer.
    IMPORTANT: mlflow.genai.load_prompt() must be called here —
    the optimizer intercepts this call to identify which prompt template to rewrite.
    """
    prompt = mlflow.genai.load_prompt(PROMPT_V1_URI)  # ← optimizer hooks in here
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": prompt.template},
            {"role": "user",   "content": text},
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content


print(f"Optimizing '{PROMPT_NAME}' using {LLM_MODEL} as the reflection model...")
print("This may take a few minutes as the optimizer runs multiple passes.\n")

opt_result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=eval_dataset,
    prompt_uris=[PROMPT_V1_URI],
    optimizer=GepaPromptOptimizer(reflection_model=f"openai:/{LLM_MODEL}"),
    scorers=[contains_keyword, is_concise],
)

OPTIMIZED_PROMPT_URI = opt_result.optimized_prompts[0].uri
optimized_version    = opt_result.optimized_prompts[0].version

mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="prod", version=optimized_version)

print(f"\n✓ Optimization complete!")
print(f"  → New version: v{optimized_version}  (aliased 'prod' → v{optimized_version})")

optimized_prompt = mlflow.genai.load_prompt(OPTIMIZED_PROMPT_URI)
print(f"\nOptimized prompt:\n{'─'*50}")
print(optimized_prompt.template)
print("─" * 50)

---
## Step 7: Re-evaluate and Compare

Run the **same dataset** through the optimized prompt and compare scores side-by-side.
Because the dataset, scorers, and inputs are identical, any score difference is purely due to the prompt change.

In [ ]:
def predict_optimized(text: str, **kwargs) -> str:
    return summarize_traced(text, OPTIMIZED_PROMPT_URI)


print("Running evaluation with optimized prompt...")
eval_optimized = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_optimized,
    scorers=[contains_keyword, is_concise],
)

print("\n── Evaluation Results: Optimized Prompt ──")
for metric, value in sorted(eval_optimized.metrics.items()):
    bar = "█" * int((value if isinstance(value, float) else 0) * 20)
    print(f"  {metric:<35} {f'{value:.0%}' if isinstance(value, float) else value}  {bar}")

In [ ]:
print("\n══════════════════════════════════════════════════")
print("   COMPARISON: v1 (baseline) vs optimized         ")
print("══════════════════════════════════════════════════")

rows = []
all_metrics = sorted(set(eval_v1.metrics) | set(eval_optimized.metrics))
for metric in all_metrics:
    v1_val  = eval_v1.metrics.get(metric)
    opt_val = eval_optimized.metrics.get(metric)
    if isinstance(v1_val, float) and isinstance(opt_val, float):
        delta  = opt_val - v1_val
        change = f"+{delta:.0%}" if delta > 0 else (f"{delta:.0%}" if delta < 0 else "no change")
        symbol = "✅" if delta > 0 else ("❌" if delta < 0 else "  ")
        rows.append({
            "Metric":    metric,
            "v1":        f"{v1_val:.0%}",
            "optimized": f"{opt_val:.0%}",
            "Change":    change,
            " ":         symbol,
        })

pd.DataFrame(rows)

---
## What's Next?

If the optimized prompt scores better, promote it to Canopy.

**Option 1 — update the prompt text in config directly:**

Copy the optimized template from the cell above and update `canopy/test/backend/config.yaml`:

```yaml
summarization:
  prompt: |  # 👈 paste optimized template here
    <optimized prompt text>
```

Commit and push — ArgoCD will redeploy Canopy with the improved prompt.

**Option 2 — promote via the Prompt Registry (if Canopy reads from MLflow):**

```python
# Pin the champion alias to the winning version
mlflow.genai.set_prompt_alias(name="summarization", alias="champion", version=optimized_version)
```

Then update `chart/values.yaml` to use `mlflow_prompt_version: champion` and push.

Either way, your feedback loop is now fully closed: real user signals → eval dataset → automated optimization → better Canopy.